In [12]:
import os 
while "src" not in os.listdir():
    os.chdir("..")
    
import numpy as np
import networkx as nx    

from src.reading_in import read_in_proteins
from wk9.spin_glass.potts import PottsNumpy

In [13]:
G = read_in_proteins(connected=True)

In [14]:
"""Demo: finding communities with PottsNumpy."""

# ---------------------------------------------------------------------------
# 1. Get a graph. Replace this with your own, e.g.
#       G = nx.read_edgelist("edges.txt")          # or read_weighted_edgelist
# Here: 6000 nodes, 30 planted communities of 200.
# ---------------------------------------------------------------------------
G = nx.planted_partition_graph(30, 200, 0.06, 0.0015, seed=1)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ---------------------------------------------------------------------------
# 2. Single run: build, anneal, inspect.
# ---------------------------------------------------------------------------
potts = PottsNumpy(G, gamma=1.0, q=100, seed=0)
history = potts.anneal(beta_start=0.05, beta_end=5.0, n_steps=80, verbose=True)

print(f"\nFound {potts.n_communities()} communities, modularity Q = {potts.modularity():.4f}")
comms = potts.communities()                      # list of sets of node ids, largest first
print("Largest community sizes:", [len(c) for c in comms[:10]])

# Sanity check: the annealing should have settled (energy flat at the end)
print(f"Energy over last 10 steps: {history[-10:].min():.1f} .. {history[-10:].max():.1f}")

# ---------------------------------------------------------------------------
# 3. Annealing is stochastic, so run several restarts and keep the lowest energy.
# ---------------------------------------------------------------------------
best = None
for seed in range(5):
    p = PottsNumpy(G, gamma=1.0, q=100, seed=seed)
    p.anneal(beta_start=0.05, beta_end=5.0, n_steps=80)
    print(f"  restart {seed}: H = {p.hamiltonian():9.1f}   "
          f"communities = {p.n_communities():3d}   Q = {p.modularity():.4f}")
    if best is None or p.hamiltonian() < best.hamiltonian():
        best = p
print(f"Best restart: H = {best.hamiltonian():.1f}, {best.n_communities()} communities")

# ---------------------------------------------------------------------------
# 4. Sweep the resolution parameter: larger gamma -> smaller communities.
# ---------------------------------------------------------------------------
print("\nResolution scan:")
for gamma in [0.25, 0.5, 1.0, 2.0, 4.0]:
    p = PottsNumpy(G, gamma=gamma, q=100, seed=0)
    p.anneal(beta_start=0.05, beta_end=5.0, n_steps=80)
    sizes = sorted((len(c) for c in p.communities()), reverse=True)
    print(f"  gamma={gamma:<5} communities={p.n_communities():3d}  "
          f"Q(gamma=1)={p.modularity():.3f}  largest sizes={sizes[:4]}")

# ---------------------------------------------------------------------------
# 5. Compare against the planted truth (only possible for synthetic graphs).
# ---------------------------------------------------------------------------
truth = np.repeat(np.arange(30), 200)
found = np.empty(len(truth), dtype=int)
for label, nodes in enumerate(best.communities()):
    found[list(nodes)] = label
# purity: fraction of nodes whose community's majority true label matches theirs
purity = sum(np.bincount(truth[found == c]).max() for c in np.unique(found)) / len(truth)
print(f"\nPurity vs planted partition: {purity:.3f}")

# ---------------------------------------------------------------------------
# 6. Use the result: write labels back onto the graph / export.
# ---------------------------------------------------------------------------
nx.set_node_attributes(G, {n: int(s) for n, s in zip(best.nodelist, best.sigma)}, "community")
# np.savetxt("labels.csv", np.c_[best.nodelist, best.sigma], fmt="%d", delimiter=",")

Graph: 6000 nodes, 62158 edges
step    0  beta=   0.050  H=     -39.137  changed= 5937  communities=100
step   10  beta=   0.090  H=     -29.424  changed= 5937  communities=100
step   20  beta=   0.160  H=     -60.176  changed= 5938  communities=100
step   30  beta=   0.287  H=    -221.681  changed= 5929  communities=100
step   40  beta=   0.515  H=    -462.875  changed= 5941  communities=100
step   50  beta=   0.922  H=    -981.066  changed= 5895  communities=100
step   60  beta=   1.652  H=   -3365.878  changed= 5454  communities=100
step   70  beta=   2.959  H=  -33726.232  changed=    5  communities=29
step   79  beta=   5.000  H=  -33728.837  changed=    0  communities=28

Found 28 communities, modularity Q = 0.5425
Largest community sizes: [400, 400, 200, 200, 200, 200, 200, 200, 200, 200]
Energy over last 10 steps: -33728.8 .. -33720.0
  restart 0: H =  -33728.8   communities =  28   Q = 0.5425
  restart 1: H =  -33799.1   communities =  29   Q = 0.5436
  restart 2: H =  -33706.

lets try more steps with larger temperature range

In [18]:
potts = PottsNumpy(G, gamma=1.0, q=200, seed=0)
history = potts.anneal(beta_start=1.0, beta_end=20.0, n_steps=2000, verbose=True)

print(f"\nFound {potts.n_communities()} communities, modularity Q = {potts.modularity():.4f}")
comms = potts.communities()                      # list of sets of node ids, largest first
print("Largest community sizes:", [len(c) for c in comms[:10]])

# Sanity check: the annealing should have settled (energy flat at the end)
print(f"Energy over last 10 steps: {history[-10:].min():.1f} -> {history[-10:].max():.1f}")


step    0  beta=   1.000  H=    -545.163  changed= 5963  communities=200
step   10  beta=   1.015  H=    -530.296  changed= 5954  communities=200
step   20  beta=   1.030  H=    -526.491  changed= 5953  communities=200
step   30  beta=   1.046  H=    -560.283  changed= 5967  communities=200
step   40  beta=   1.062  H=    -620.534  changed= 5958  communities=200
step   50  beta=   1.078  H=    -578.187  changed= 5956  communities=200
step   60  beta=   1.094  H=    -576.522  changed= 5957  communities=200
step   70  beta=   1.111  H=    -611.251  changed= 5955  communities=200
step   80  beta=   1.127  H=    -617.795  changed= 5956  communities=200
step   90  beta=   1.144  H=    -699.721  changed= 5955  communities=200
step  100  beta=   1.162  H=    -621.848  changed= 5956  communities=200
step  110  beta=   1.179  H=    -706.171  changed= 5949  communities=200
step  120  beta=   1.197  H=    -739.613  changed= 5951  communities=200
step  130  beta=   1.215  H=    -723.866  changed= 

In [ ]:
louvain_communities = nx.community.louvain_communities(G, resolution=1.0)